# NeuroTrain Lab — Notebook 3: Optimizadores

**Tema:** cómo un optimizador usa el gradiente (Notebook 2) para de verdad mover
los weights, qué papel juega la tasa de aprendizaje (*learning rate*), en qué se
diferencian SGD, Momentum y Adam, y por qué a veces los gradientes "desaparecen"
en redes profundas.

> Notebook 3 de 4. Ya sabes calcular el gradiente de la pérdida. Ahora usamos ese
> gradiente para **dar pasos** que reduzcan el error — y veremos que no todos los
> pasos son iguales.

## 🎯 Qué aprenderás en este notebook

1. Qué hace el descenso de gradiente, paso a paso, sobre una función de pérdida.
2. Por qué la tasa de aprendizaje es la decisión más delicada de todo el entrenamiento.
3. Qué añade Momentum sobre SGD puro, y qué hace Adam distinto (a nivel conceptual).
4. Cómo reconocer un gradiente que se desvanece en una red profunda, y por qué pasa.
5. Un diagnóstico rápido para "mi red no aprende" antes de tocar el código a ciegas.

**Mapa mental:** `gradiente → paso de descenso → learning rate → SGD → Momentum → Adam → gradientes que desaparecen`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_moons

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurotrain.celebrations import celebrate
from neurotrain.visualization import plot_gradient_magnitude_by_layer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("NumPy:", np.__version__, "| TensorFlow:", tf.__version__)
print("Raíz del proyecto:", PROJECT_ROOT)

## 0. Dónde estamos

En el Notebook 2 aprendiste a calcular $\partial L / \partial w$ para cada weight
de la red — el gradiente te dice **en qué dirección** crece la pérdida. Lo único
que falta es la regla que decide, con ese gradiente en la mano, **cuánto y cómo**
mover cada weight. Esa regla es el **optimizador**.

## 1. Descenso de gradiente: el excursionista en la niebla

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — Un excursionista que no ve el valle</b><br><br>
Imagina a alguien bajando una montaña envuelta en niebla espesa. No puede ver el
valle ni el mapa completo — solo siente, bajo sus pies, hacia dónde baja el
terreno **en ese punto exacto**. Da un paso en esa dirección, vuelve a sentir la
pendiente, da otro paso, y así sucesivamente.

Eso es exactamente el descenso de gradiente: en cada punto solo conocemos el
gradiente **local** (la pendiente bajo nuestros pies), no la forma completa de la
función de pérdida. Actualizamos así:

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

donde $\eta$ (eta) es la **tasa de aprendizaje** — el tamaño del paso del excursionista.
</div>

In [ ]:
def f(w):
    """Una 'pérdida' de juguete en 1D: un cuenco con mínimo en w=3."""
    return (w - 3) ** 2 + 1


def gradiente(w):
    return 2 * (w - 3)


def descenso_gradiente(w_inicial, lr, pasos):
    trayectoria = [w_inicial]
    w = w_inicial
    for _ in range(pasos):
        w = w - lr * gradiente(w)
        trayectoria.append(w)
    return np.array(trayectoria)


trayectoria = descenso_gradiente(w_inicial=-2.0, lr=0.2, pasos=15)
print("Posiciones de w:", trayectoria.round(3))
print("Pérdida final:", round(f(trayectoria[-1]), 4), "(el mínimo real vale 1.0 en w=3)")

In [ ]:
w_curva = np.linspace(-3, 8, 200)
plt.figure(figsize=(7, 4.5))
plt.plot(w_curva, f(w_curva), color="#94A3B8", label="f(w) = (w-3)² + 1")
plt.plot(trayectoria, f(trayectoria), "o-", color="#7C3AED", label="pasos del descenso")
for i in range(len(trayectoria) - 1):
    plt.annotate(
        "", xy=(trayectoria[i + 1], f(trayectoria[i + 1])),
        xytext=(trayectoria[i], f(trayectoria[i])),
        arrowprops=dict(arrowstyle="->", color="#F97316", alpha=0.6),
    )
plt.scatter([3], [1], color="#22C55E", zorder=5, label="mínimo real (w=3)")
plt.title("El excursionista bajando el cuenco de pérdida, paso a paso")
plt.xlabel("w")
plt.ylabel("f(w)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### ✏️ Ejercicio

Completa `descenso_gradiente_manual`: en cada paso, calcula el gradiente en el `w`
actual y actualízalo restando `lr * gradiente`. Pruébalo con `w_inicial=6.0, lr=0.3, pasos=10`
y confirma que `w` se acerca a 3.

In [ ]:
def descenso_gradiente_manual(w_inicial, lr, pasos):
    w = w_inicial
    for _ in range(pasos):
        g = gradiente(w)
        w = w - ✏️✏️✏️
    return w


w_final = descenso_gradiente_manual(w_inicial=6.0, lr=0.3, pasos=10)
print("w final:", round(w_final, 4))

<details>
<summary><b>Ver solución</b></summary>

```python
def descenso_gradiente_manual(w_inicial, lr, pasos):
    w = w_inicial
    for _ in range(pasos):
        g = gradiente(w)
        w = w - lr * g
    return w


w_final = descenso_gradiente_manual(w_inicial=6.0, lr=0.3, pasos=10)
print("w final:", round(w_final, 4))
```

</details>

## 2. El efecto de la tasa de aprendizaje

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ DUDA PROBABLE — ¿Por qué no usar siempre un learning rate gigante para llegar rápido?</b><br><br>
Porque el excursionista, si da pasos demasiado grandes, puede saltar por encima
del valle y aterrizar en la ladera de enfrente — o incluso más lejos de donde
empezó. Un paso pequeño es seguro pero lento; un paso grande es rápido pero puede
hacerte rebotar sin parar, o divergir. Vamos a verlo con números.
</div>

In [ ]:
escenarios = {
    "lr demasiado pequeño (0.01)": descenso_gradiente(-2.0, lr=0.01, pasos=40),
    "lr adecuado (0.3)": descenso_gradiente(-2.0, lr=0.3, pasos=40),
    "lr demasiado grande (1.05)": descenso_gradiente(-2.0, lr=1.05, pasos=40),
}

plt.figure(figsize=(8, 4.5))
colores = ["#2563EB", "#22C55E", "#F97316"]
for (nombre, trayectoria_e), color in zip(escenarios.items(), colores):
    plt.plot(trayectoria_e, marker=".", label=nombre, color=color)
plt.axhline(3, color="#94A3B8", linestyle="--", linewidth=0.8, label="mínimo (w=3)")
plt.title("Mismo punto de partida, tres learning rates distintos")
plt.xlabel("paso")
plt.ylabel("w")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

for nombre, trayectoria_e in escenarios.items():
    print(f"{nombre}: w tras 40 pasos = {trayectoria_e[-1]:.3f}")

Con `lr=0.01` el excursionista apenas se ha movido tras 40 pasos: sigue lejos del
mínimo. Con `lr=0.3` converge de forma suave. Con `lr=1.05` cada paso lo manda
**más lejos** que el anterior — está divergiendo, no convergiendo.

<div style="border-left:4px solid #F97316; background:#FFF7ED; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>⚠️ ERROR TÍPICO — Un learning rate demasiado alto no siempre se nota a simple vista al principio</b><br><br>
En una red real, un learning rate demasiado alto a veces produce una pérdida que
**baja un poco al inicio** y luego empieza a oscilar o se dispara a `NaN`. No
asumas que "si arrancó bajando, el learning rate está bien": sigue vigilando la
curva varias épocas más.
</div>

### ✏️ Ejercicio

Elige un cuarto learning rate para probar (cualquier número positivo que no sea
0.01, 0.3 o 1.05). **Antes de ejecutar la celda**, escribe en un comentario qué
crees que va a pasar (converge suave / lento / diverge). Luego ejecuta y comprueba
si acertaste.

In [ ]:
# Mi predicción: ✏️✏️✏️ (escribe aquí converge suave / demasiado lento / diverge)
mi_lr = 0.6
trayectoria_mia = descenso_gradiente(-2.0, lr=mi_lr, pasos=40)
print("w tras 40 pasos:", round(trayectoria_mia[-1], 3))
plt.plot(trayectoria_mia, marker=".", color="#7C3AED")
plt.axhline(3, color="#94A3B8", linestyle="--")
plt.title(f"Mi learning rate = {mi_lr}")
plt.show()

<details>
<summary><b>Ver solución</b></summary>

```python
# Mi predicción: con lr=0.6 el paso es mayor que el óptimo pero sigue dentro del
# rango que converge (0 < lr < 1 para esta parábola); esperaría oscilación
# amortiguada que sí llega cerca del mínimo.
mi_lr = 0.6
trayectoria_mia = descenso_gradiente(-2.0, lr=mi_lr, pasos=40)
print("w tras 40 pasos:", round(trayectoria_mia[-1], 3))
plt.plot(trayectoria_mia, marker=".", color="#7C3AED")
plt.axhline(3, color="#94A3B8", linestyle="--")
plt.title(f"Mi learning rate = {mi_lr}")
plt.show()
```

</details>

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🔥 Mitad del camino: de aquí a nada desentrañarás los misterios de la mente artificial.</div>

## 3. SGD, Momentum y Adam: el mismo MLP, tres optimizadores

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — Momentum: una bola que conserva algo de su velocidad</b><br><br>
El excursionista de la Sección 1 decide su paso mirando **solo** la pendiente
actual, como si empezara de cero en cada paso. **Momentum** es distinto: imagina
en vez de un excursionista, una bola rodando cuesta abajo — conserva parte de la
velocidad que ya traía. Eso le permite atravesar pequeños baches sin frenarse del
todo, y acelerar en tramos donde la pendiente apunta siempre en la misma dirección.

**Adam** va un paso más allá: adapta el tamaño del paso **por cada parámetro por
separado**, usando estadísticas acumuladas del gradiente reciente. No necesitas
memorizar su fórmula — solo saber que combina la idea de Momentum con pasos
adaptativos, y por eso suele converger rápido "de fábrica" sin apenas ajustar el
learning rate a mano.
</div>

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=RANDOM_STATE)

plt.figure(figsize=(4.5, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="RdBu_r", edgecolor="white")
plt.title("make_moons (más ruido que en el Notebook 1): el reto para los 3 optimizadores")
plt.show()

In [ ]:
def construir_mlp():
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])


optimizadores = {
    "SGD": tf.keras.optimizers.SGD(learning_rate=0.05),
    "SGD + Momentum": tf.keras.optimizers.SGD(learning_rate=0.05, momentum=0.9),
    "Adam": tf.keras.optimizers.Adam(learning_rate=0.05),
}

historiales = {}
for nombre, optimizador in optimizadores.items():
    modelo = construir_mlp()
    modelo.compile(optimizer=optimizador, loss="binary_crossentropy")
    historial = modelo.fit(X_moons, y_moons, epochs=50, verbose=0)
    historiales[nombre] = historial.history["loss"]
    print(f"{nombre:16s} -> pérdida inicial {historial.history['loss'][0]:.3f}, "
          f"pérdida final {historial.history['loss'][-1]:.3f}")

In [ ]:
plt.figure(figsize=(7.5, 4.5))
colores_opt = {"SGD": "#F97316", "SGD + Momentum": "#2563EB", "Adam": "#22C55E"}
for nombre, perdidas in historiales.items():
    plt.plot(perdidas, label=nombre, color=colores_opt[nombre])
plt.title("Misma red, mismos datos, mismo learning rate: solo cambia el optimizador")
plt.xlabel("época")
plt.ylabel("pérdida (binary cross-entropy)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

Con el mismo `learning_rate=0.05` para los tres, verás que SGD puro es el que más
tarda en bajar la pérdida, Momentum acelera esa bajada, y Adam suele converger más
rápido y de forma más estable desde las primeras épocas — sin que hayamos tocado
ningún otro hiperparámetro.

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 PARA RECORDAR — Adam no es magia, es la opción por defecto razonable</b><br><br>
Adam suele funcionar bien "tal cual" en muchos problemas, por eso es tan popular
como punto de partida. Pero "por defecto razonable" no es "siempre óptimo": para
problemas concretos, SGD + Momentum bien ajustado a veces generaliza mejor. Elegir
optimizador sigue siendo una decisión que se valida, no una ley fija.
</div>

## 4. Gradientes que desaparecen: por qué la profundidad no es gratis

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — Multiplicar muchos números pequeños, muchas veces</b><br><br>
En el Notebook 2 viste que backpropagation usa la **regla de la cadena**: el
gradiente de una capa profunda se calcula multiplicando, capa a capa, las
derivadas locales de todas las capas que hay entre esa capa y la salida. La
derivada de Sigmoid nunca supera **0.25** (su pico está en $z=0$). Si encadenas 8-10
capas con Sigmoid, estás multiplicando 8-10 números que valen como mucho 0.25 —
el resultado se hace minúsculo muy rápido. ReLU, en cambio, tiene derivada 1 (para
las neuronas activas) o 0, así que no aplasta el gradiente de la misma forma al
encadenarse.
</div>

In [ ]:
def construir_red_profunda(activacion, profundidad=9):
    capas = [tf.keras.layers.Input(shape=(2,))]
    for _ in range(profundidad):
        capas.append(tf.keras.layers.Dense(16, activation=activacion))
    capas.append(tf.keras.layers.Dense(1, activation="sigmoid"))
    return tf.keras.Sequential(capas)


def magnitudes_de_gradiente(modelo, X, y):
    """Media de |gradiente| del kernel de cada capa Dense, de salida a entrada."""
    X_t = tf.convert_to_tensor(X, dtype=tf.float32)
    y_t = tf.convert_to_tensor(y.reshape(-1, 1), dtype=tf.float32)
    with tf.GradientTape() as tape:
        prediccion = modelo(X_t, training=True)
        perdida = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_t, prediccion))
    gradientes = tape.gradient(perdida, modelo.trainable_weights)
    kernels = [g for w, g in zip(modelo.trainable_weights, gradientes) if "kernel" in w.name]
    magnitudes = [float(tf.reduce_mean(tf.abs(g))) for g in kernels]
    magnitudes.reverse()  # de la capa más cercana a la salida hacia la entrada
    return magnitudes


magnitudes_por_activacion = {}
for activacion in ["sigmoid", "relu"]:
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    red = construir_red_profunda(activacion)
    magnitudes_por_activacion[activacion] = magnitudes_de_gradiente(red, X_moons, y_moons)
    print(activacion, [f"{m:.2e}" for m in magnitudes_por_activacion[activacion]])

In [ ]:
plot_gradient_magnitude_by_layer(magnitudes_por_activacion, lang="es")
plt.show()

Con esta arquitectura exacta (9 capas ocultas de 16 neuronas, semilla 42), la capa
Sigmoid más cercana a la salida tiene un gradiente medio de **~4.0 × 10⁻²**, y la
capa más cercana a la entrada cae hasta **~1.6 × 10⁻⁸** — una caída de **más de 6
órdenes de magnitud**. Con ReLU, todas las capas se mantienen en un rango parecido,
entre **~1.6 × 10⁻⁴ y ~9.4 × 10⁻⁴** sin tendencia a desaparecer. Esa capa más
cercana a la entrada en la red Sigmoid recibe un gradiente tan diminuto que,
prácticamente, **deja de aprender**: sus weights apenas cambian entrenamiento tras
entrenamiento.

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ DUDA PROBABLE — ¿Entonces nunca debo usar Sigmoid en capas ocultas?</b><br><br>
Como regla práctica: usa ReLU (o variantes como Leaky ReLU) en las **capas
ocultas** de redes profundas, y reserva Sigmoid para la **capa de salida** en
clasificación binaria, donde solo hay una capa y el problema no aparece. Es
justo lo que ya venías haciendo desde el Notebook 1 — ahora sabes el porqué.
</div>

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🚀 La red está tomando forma bajo tus manos.</div>

## 5. '¿Por qué mi red no aprende?' — tabla de diagnóstico rápido

| Síntoma | Causa probable | Qué revisar |
|---|---|---|
| La pérdida está plana desde la época 1 | Learning rate demasiado pequeño / gradiente desvanecido / neuronas ReLU "muertas" | Sube el learning rate; revisa la magnitud del gradiente por capa (Sección 4); comprueba cuántas activaciones ReLU dan siempre 0 |
| La pérdida explota o se vuelve `NaN` | Learning rate demasiado alto / inestabilidad numérica | Baja el learning rate; revisa si hay valores de entrada sin normalizar o `logits` extremos |
| La pérdida de entrenamiento mejora pero la de validación empeora | Sobreajuste (el modelo memoriza en vez de generalizar) | No se cubre aquí — es el tema central del Notebook 4 |
| La pérdida baja muy despacio, de forma constante | Learning rate algo bajo, u optimizador sin momentum en un terreno con curvatura difícil | Prueba Momentum o Adam (Sección 3) antes de tocar la arquitectura |
| La pérdida oscila sin bajar de forma clara | Learning rate demasiado alto para ese optimizador | Reduce el learning rate; compara con la Sección 2 |

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 PARA RECORDAR — Antes de cambiar la arquitectura, revisa el optimizador</b><br><br>
Es tentador, cuando una red no aprende, añadir capas o neuronas de inmediato. Pero
muchos de los síntomas más comunes (pérdida plana, pérdida que explota, convergencia
lentísima) tienen que ver con el **learning rate** o el **optimizador**, no con el
tamaño de la red. Revisa primero lo barato de cambiar.
</div>

## 🎯 Autoevaluación

Respóndelas sin mirar atrás. No necesitas frases perfectas: explica el mecanismo con tus palabras.

**1. ¿Qué añade Momentum sobre el descenso de gradiente (SGD) puro?**

A. Nada, son matemáticamente idénticos
B. Conserva parte de la 'velocidad' de pasos anteriores, suavizando y acelerando la trayectoria
C. Elimina por completo la necesidad de elegir un learning rate
D. Solo funciona si la red tiene una única capa

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Momentum acumula una fracción del paso anterior, como una bola rodando que conserva velocidad, en vez de decidir cada paso solo con la pendiente actual.

</details>

**2. En un gráfico de magnitud de gradiente por capa (como el de la Sección 4), ¿cómo se reconoce un problema de gradientes que desaparecen?**

A. Todas las capas tienen una magnitud similar, sin caer con la profundidad
B. La magnitud cae varios órdenes de magnitud a medida que nos alejamos de la capa de salida
C. La magnitud sube exponencialmente cerca de la entrada
D. El gráfico no tiene relación con el problema; hay que mirar la pérdida

<details>
<summary><b>Ver respuesta</b></summary>

**B.** El patrón característico es una caída pronunciada (multiplicativa, por eso se ve en escala log) de la magnitud del gradiente en las capas más cercanas a la entrada.

</details>

**3. Según la tabla de diagnóstico de la Sección 5, ¿qué deberías sospechar primero si la pérdida de entrenamiento y de validación se separan (la de validación empeora)?**

A. Un learning rate demasiado alto
B. Sobreajuste — tema que se desarrolla en el Notebook 4
C. Un gradiente desvanecido
D. Un error de tipeo en la función de pérdida

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Train mejorando mientras validación empeora es la firma clásica de sobreajuste, que este notebook solo señala — se trabaja en profundidad en el Notebook 4.

</details>

**4. En el experimento de la Sección 2, ¿qué le pasó a la trayectoria con `lr=1.05`?**

A. Convergió más rápido que con `lr=0.3`
B. Se quedó exactamente en el punto de partida
C. Cada paso se alejó más del mínimo que el anterior: diverge
D. Llegó al mínimo pero osciló ligeramente alrededor de él

<details>
<summary><b>Ver respuesta</b></summary>

**C.** Con un learning rate mayor que el rango estable para esta parábola, cada actualización sobrepasa el mínimo por un margen creciente: la trayectoria diverge en vez de converger.

</details>

**5. ¿Qué hace Adam de forma distinta a Momentum, a nivel conceptual (sin fórmulas)?**

A. Adam no usa learning rate en absoluto
B. Adam adapta el tamaño del paso por cada parámetro individualmente, usando estadísticas acumuladas del gradiente
C. Adam solo sirve para clasificación con Softmax
D. Adam es idéntico a SGD sin momentum

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Adam combina la idea de Momentum con pasos adaptativos por parámetro, calculados a partir de estadísticas acumuladas de gradientes recientes (media y varianza, a grandes rasgos).

</details>

**6. ¿Por qué crees que Adam se convirtió en la opción por defecto en la industria? Explícalo con tus propias palabras.**

<details>
<summary><b>Qué debería incluir una buena respuesta</b></summary>

- Menciona que suele converger rápido sin apenas ajustar hiperparámetros a mano.
- Conecta la idea de pasos adaptativos por parámetro con problemas donde distintos weights necesitan distinta escala de actualización.
- Reconoce que 'por defecto razonable' no significa 'siempre el mejor' — sigue siendo una elección que se valida.

</details>

**7. Un compañero de equipo te dice: 'la pérdida de mi red está plana desde la época 1'. Describe, paso a paso, cómo lo diagnosticarías.**

<details>
<summary><b>Qué debería incluir una buena respuesta</b></summary>

- Empieza revisando el learning rate (¿demasiado pequeño?) antes de tocar la arquitectura.
- Menciona revisar la magnitud del gradiente por capa para descartar un gradiente desvanecido.
- Contempla la posibilidad de neuronas ReLU muertas o datos de entrada sin normalizar.
- Sigue un orden razonado en vez de cambiar cosas al azar, apoyándose en la tabla de diagnóstico de la Sección 5.

</details>

In [ ]:
celebrate(
    "🎉 ¡Enhorabuena! Completaste el Notebook 3: Optimizadores 🎉",
    "Ya sabes cómo un optimizador usa el gradiente para dar pasos, qué aporta cada "
    "uno de SGD/Momentum/Adam, y cómo diagnosticar una red que no aprende. En el "
    "Notebook 4 entrenamos de verdad: épocas, batches y cómo evitar el sobreajuste.",
)